In [1]:
from pypdf import PdfReader
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from bs4 import BeautifulSoup as bs
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import joblib

## Extract PDF file

In [2]:
reader = PdfReader("python.pdf")

## Load Text format

In [3]:
text = ""
for page in reader.pages:
    text += page.extract_text()

In [4]:
print(text)

Think Python
How to Think Like a Computer Scientist
2nd Edition, Version 2.4.0Think Python
How to Think Like a Computer Scientist
2nd Edition, Version 2.4.0
Allen Downey
Green Tea Press
Needham, MassachusettsCopyright © 2015 Allen Downey.
Green Tea Press
9 Washburn Ave
Needham MA 02492
Permission is granted to copy, distribute, and/or modify this document under the terms of the
Creative Commons Attribution-NonCommercial 3.0 Unported License, which is available at http:
//creativecommons.org/licenses/by-nc/3.0/.
The original form of this book is LATEX source code. Compiling this LATEX source has the effect of gen-
erating a device-independent representation of a textbook, which can be converted to other formats
and printed.
The LATEX source for this book is available from http://www.thinkpython.comPreface
The strange history of this book
In January 1999 I was preparing to teach an introductory programming class in Java. I had
taught it three times and I was getting frustrated. The failu

## NLP Text Normalization

In [5]:
def textNormalize(text):
    text = text.lower()
    text = bs(text,"html.parser").get_text()
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)
    text = text.translate(str.maketrans('','',string.punctuation))
    text = re.sub(r"\d+",'',text)
    text = re.sub(r'''[^\w\s.,;:!?''""\-]'''," ",text)
    text = re.sub(r"\s+",' ',text).strip()

    return text

In [6]:
text = textNormalize(text)

In [7]:
print(text)

think python how to think like a computer scientist nd edition version think python how to think like a computer scientist nd edition version allen downey green tea press needham massachusettscopyright allen downey green tea press washburn ave needham ma permission is granted to copy distribute andor modify this document under the terms of the creative commons attributionnoncommercial unported license which is available at http creativecommonsorglicensesbync the original form of this book is latex source code compiling this latex source has the effect of gen erating a deviceindependent representation of a textbook which can be converted to other formats and printed the latex source for this book is available from the strange history of this book in january i was preparing to teach an introductory programming class in java i had taught it three times and i was getting frustrated the failure rate in the class was too high and even for students who succeeded the overall level of achieveme

## Recursive Chunk Split

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)

In [9]:
chapters = text.split("chapter")
chunks = []
for chapter in chapters:
    docs = text_splitter.create_documents([chapter])
    chunks.extend([doc.page_content for doc in docs])

print(len(chunks))

676


In [10]:
print(len(chunks))

676


In [11]:
for i in range(3):
    print(f"Chunk {i+1}")
    print(chunks[i])
    print("=" * 100)

Chunk 1
think python how to think like a computer scientist nd edition version think python how to think like a computer scientist nd edition version allen downey green tea press needham massachusettscopyright allen downey green tea press washburn ave needham ma permission is granted to copy distribute andor modify this document under the terms of the creative commons attributionnoncommercial unported license which is available at http creativecommonsorglicensesbync the original form of this book is latex source code compiling this latex source has the effect of gen erating a deviceindependent representation of a textbook which can be converted to other formats and printed the latex source for this book is available from the strange history of this book in january i was preparing to teach an
Chunk 2
to other formats and printed the latex source for this book is available from the strange history of this book in january i was preparing to teach an introductory programming class in java 

## Embeddings

In [12]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
embeddings = embedding_model.encode(chunks,convert_to_numpy=True,show_progress_bar=True)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [14]:
print(embeddings.shape)

(676, 384)


## FAISS vector db

In [15]:
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(index.ntotal)

676


In [16]:
print(index.ntotal)

676


## store external file 

In [17]:
joblib.dump(chunks, "chunks.pkl")
faiss.write_index(index, "faiss_index.index")